# Live MCT Core PCE Inflation Volatility

This standalone Python notebook pulls the latest revised monthly sector data directly from the U.S. Bureau of Economic Analysis (BEA) and estimates core PCE inflation volatility with the same multivariate core trend (MCT) model structure used by the Federal Reserve Bank of New York:

- common and sector-specific random-walk trends;
- time-varying common loadings;
- common stochastic volatility;
- sector-specific stochastic volatility with MA(3) shocks; and
- the published discrete outlier mixture.

There is no NY Fed data file, October 2023 reference fit, MATLAB validation, synthetic test, class, virtual-environment reference, run mode, or data mode. Everything runs from top to bottom in a regular Python Jupyter kernel.

The speedup comes from equivalent linear-algebra operations, a shorter MCMC run, no thinning, and retaining only the total/common/sector-specific volatility quantities used in the final measure. On the machine used to benchmark the earlier version, the default 750 iterations took roughly 45 minutes. Runtime will vary with the sample length and computer.

In [ ]:
%pip install -q numpy pandas requests matplotlib scipy

In [1]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy.special import logsumexp
from scipy.stats import invgamma

warnings.filterwarnings("ignore",category=RuntimeWarning)
plt.style.use("seaborn-v0_8-whitegrid")

## Settings

The BEA key is hardcoded as requested. Do not publish or commit this notebook to a public repository while the key remains in it.

The structural model and priors match the MCT approach. The shorter sampler reduces Monte Carlo precision, but not the model specification. For a longer publication-style run, change `BURN` and `DRAWS` to `3000`.

In [2]:
BEA_API_KEY="FD4E7F46-5D89-4CA8-BE48-4F5997F83B3B"

SEED=2022
BURN=50  ##250
DRAWS=100  ##500
START_DATE=pd.Timestamp("1960-01-01")

OUTPUT=Path.cwd()/"outputs"/"mct_core_pce_volatility_fast"
for folder in ("data","estimates","figures"):
    (OUTPUT/folder).mkdir(parents=True,exist_ok=True)
print("Output:",OUTPUT)

Output: c:\Users\abour\OneDrive\Economic Research\Inflation Vol\outputs\mct_core_pce_volatility_fast


## Pull and construct the 17-sector core PCE input from BEA

Price indexes come from BEA table `U20404`; nominal expenditures come from `U20405`. Food at home, gasoline and other energy goods, and gas/electric utilities remain in the 17-sector model but receive zero weight in the core PCE aggregation.

In [3]:
SECTORS = [
    ("DMOTRG","Motor vehicles and parts","DMOTRG","DMOTRC",1),
    ("DFDHRG","Furnishings and durable household equipment","DFDHRG","DFDHRC",1),
    ("DREQRG","Recreational goods and vehicles","DREQRG","DREQRC",1),
    ("DODGRG","Other durable goods","DODGRG","DODGRC",1),
    ("DFXARG","Food and beverages purchased for off-premises consumption","DFXARG","DFXARC",0),
    ("DCLORG","Clothing and footwear","DCLORG","DCLORC",1),
    ("DGOERG","Gasoline and other energy goods","DGOERG","DGOERC",0),
    ("DONGRG","Other nondurable goods","DONGRG","DONGRC",1),
    ("pce_hu_xes","Housing excluding gas and electric utilities",None,None,1),
    ("pce_hu_es","Gas and electric utilities","DELGRG","DELGRC",0),
    ("DHLCRG","Health care","DHLCRG","DHLCRC",1),
    ("DTRSRG","Transportation services","DTRSRG","DTRSRC",1),
    ("DRCARG","Recreation services","DRCARG","DRCARC",1),
    ("DFSARG","Food services and accommodations","DFSARG","DFSARC",1),
    ("DIFSRG","Financial services and insurance","DIFSRG","DIFSRC",1),
    ("DOTSRG","Other services","DOTSRG","DOTSRC",1),
    ("DNPIRG","Final consumption expenditures of nonprofit institutions serving households (NPISHs)","DNPIRG","DNPIRC",1),
]

SECTOR_IDS=[row[0] for row in SECTORS]
SECTOR_LABELS=[row[1] for row in SECTORS]
CORE=np.array([bool(row[4]) for row in SECTORS])
BEA_URL="https://apps.bea.gov/api/data"

def bea_table(table_name):
    parameters={
        "UserID":BEA_API_KEY,
        "method":"GetData",
        "DataSetName":"NIUnderlyingDetail",
        "TableName":table_name,
        "Frequency":"M",
        "Year":"X",
        "ResultFormat":"JSON",
    }
    response=requests.get(BEA_URL,params=parameters,timeout=240)
    response.raise_for_status()
    results=response.json()["BEAAPI"]["Results"]
    if "Error" in results:
        raise RuntimeError(f"BEA API error: {results['Error']}")
    rows=pd.DataFrame(results["Data"])
    rows["date"]=pd.to_datetime(rows["TimePeriod"].str.replace("M","-",regex=False)+"-01")
    rows["value"]=pd.to_numeric(rows["DataValue"].str.replace(",","",regex=False),errors="coerce")
    return rows

def annualized_change(price_index):
    return 100*((price_index/price_index.shift(1))**12-1)

def tornqvist_index(price_indexes,nominal_expenditures):
    price_indexes,nominal_expenditures=price_indexes.align(nominal_expenditures,join="inner",axis=0)
    shares=nominal_expenditures.to_numpy(float)
    shares=shares/shares.sum(axis=1,keepdims=True)
    average_shares=.5*(shares+np.vstack([np.full((1,shares.shape[1]),np.nan),shares[:-1]]))
    log_changes=np.log(price_indexes.to_numpy(float)/price_indexes.shift(1).to_numpy(float))
    log_growth=pd.Series(np.sum(average_shares*log_changes,axis=1),index=price_indexes.index)
    level=100*np.exp(log_growth.fillna(0).cumsum())
    level[log_growth.isna()]=np.nan
    level.iloc[0]=100
    return level

def load_bea_data():
    price_rows=bea_table("U20404")
    nominal_rows=bea_table("U20405")
    raw_prices=price_rows.pivot_table(index="date",columns="SeriesCode",values="value",aggfunc="first")
    raw_nominal=nominal_rows.pivot_table(index="date",columns="SeriesCode",values="value",aggfunc="first")

    prices=pd.DataFrame(index=raw_prices.index)
    nominal=pd.DataFrame(index=raw_nominal.index)
    for sector_id,unused_label,price_code,nominal_code,unused_core in SECTORS:
        if sector_id=="pce_hu_xes":
            prices[sector_id]=tornqvist_index(
                raw_prices[["DHSGRG","DWRSRG"]],
                raw_nominal[["DHSGRC","DWRSRC"]],
            )
            nominal[sector_id]=raw_nominal[["DHSGRC","DWRSRC"]].sum(axis=1,min_count=2)
        else:
            prices[sector_id]=raw_prices[price_code]
            nominal[sector_id]=raw_nominal[nominal_code]

    prices,nominal=prices.align(nominal,join="inner",axis=0)
    inflation=annualized_change(prices)
    complete=inflation.notna().all(axis=1)&nominal.notna().all(axis=1)&(nominal>0).all(axis=1)
    prices=prices.loc[complete]
    nominal=nominal.loc[complete]
    inflation=inflation.loc[complete]

    all_weights=nominal.div(nominal.sum(axis=1),axis=0)
    core_nominal=nominal.copy()
    core_nominal.loc[:,~CORE]=0
    core_weights=core_nominal.div(core_nominal.sum(axis=1),axis=0)

    return {
        "dates":inflation.index,
        "inflation":inflation.to_numpy(float),
        "prices":prices.to_numpy(float),
        "nominal":nominal.to_numpy(float),
        "weights":all_weights.to_numpy(float),
        "core_weights":core_weights.to_numpy(float),
        "vintage":inflation.index[-1].strftime("%Y-%m"),
        "retrieved_utc":pd.Timestamp.now(tz="UTC").isoformat(),
    }

In [4]:
data=load_bea_data()
use=data["dates"]>=START_DATE
dates=data["dates"][use]
y=data["inflation"][use]
w=data["core_weights"][use]

if y.shape[1]!=17 or not np.isfinite(y).all():
    raise RuntimeError("The BEA sector input is incomplete.")
if not np.isfinite(w).all() or (w<0).any():
    raise RuntimeError("Core PCE weights are invalid.")
if np.max(np.abs(w.sum(axis=1)-1))>1e-10:
    raise RuntimeError("Core PCE weights do not sum to one.")
if np.max(np.abs(w[:,~CORE]))>1e-14:
    raise RuntimeError("Food and energy sectors must have zero core weight.")

rows=[]
for j,(sector_id,sector_label,unused_price,unused_nominal,in_core) in enumerate(SECTORS):
    rows.append(pd.DataFrame({
        "date":dates,
        "sector_id":sector_id,
        "sector_label":sector_label,
        "annualized_monthly_inflation":y[:,j],
        "price_index":data["prices"][use,j],
        "nominal_expenditure_saar_millions":data["nominal"][use,j],
        "all_pce_weight":data["weights"][use,j],
        "core_pce_weight":w[:,j],
        "included_in_core":bool(in_core),
    }))
sector_data=pd.concat(rows,ignore_index=True)
sector_data["data_vintage"]=data["vintage"]
sector_data["retrieved_utc"]=data["retrieved_utc"]
sector_data.to_csv(OUTPUT/"data"/"sector_data.csv",index=False)

print(f"{len(dates)} months and {y.shape[1]} sectors: {dates[0].date()} to {dates[-1].date()}")
print("BEA data vintage:",data["vintage"])

798 months and 17 sectors: 1960-01-01 to 2026-06-01
BEA data vintage: 2026-06


## MCT numerical routines

In [ ]:
OP=np.array([.00609,.04775,.13057,.20674,.22715,.18842,.12047,.05591,.01575,.00115])
OM=np.array([1.92677,1.34744,.73504,.02266,-.85173,-1.97278,-3.46788,-5.55246,-8.68384,-14.65])
OV=np.array([.11265,.17788,.26768,.40611,.62699,.98583,1.57469,2.54498,4.16591,7.33342])
SGRID=np.r_[1.,np.linspace(2,10,39)]

def sym(A): return (A+A.T)/2
def mvdraw(m,C,rng):
    C=sym(C)
    eye=np.eye(len(m))
    for jitter in (1e-10,1e-8,1e-6):
        try:
            L=np.linalg.cholesky(C+jitter*eye)
            return m+L@rng.standard_normal(len(m))
        except np.linalg.LinAlgError:
            pass
    values,vectors=np.linalg.eigh(C)
    return m+vectors@(np.sqrt(np.maximum(values,1e-12))*rng.standard_normal(len(m)))

def solve_sym(A,B):
    A=sym(A)
    eye=np.eye(A.shape[0])
    for jitter in (1e-10,1e-8,1e-6):
        try:
            return np.linalg.solve(A+jitter*eye,B)
        except np.linalg.LinAlgError:
            pass
    return np.linalg.pinv(A)@B

def rwffbs(obs,var,sd,rng,m0=0.,C0=100.):
    T=len(obs); m=np.zeros(T); C=np.zeros(T); a=np.zeros(T); R=np.zeros(T)
    pm=float(m0); pc=float(C0); q=max(sd*sd,1e-10)
    for t in range(T):
        a[t]=pm; R[t]=pc+q
        if np.isfinite(obs[t]) and np.isfinite(var[t]):
            k=R[t]/(R[t]+max(var[t],1e-10)); m[t]=a[t]+k*(obs[t]-a[t]); C[t]=max((1-k)*R[t],1e-12)
        else: m[t],C[t]=a[t],R[t]
        pm,pc=m[t],C[t]
    x=np.empty(T); x[-1]=rng.normal(m[-1],np.sqrt(C[-1]))
    for t in range(T-2,-1,-1):
        b=C[t]/max(R[t+1],1e-12); mu=m[t]+b*(x[t+1]-a[t+1]); vv=max(C[t]-b*b*R[t+1],1e-12)
        x[t]=rng.normal(mu,np.sqrt(vv))
    return x

def update_sv(x,sigma,gamma,rng,skip,C0):
    z=np.log(np.asarray(x)**2+1e-3); h=2*np.log(np.maximum(sigma,1e-8))
    lp=np.log(OP)[None,:]-.5*np.log(OV)[None,:]-.5*(z[:,None]-h[:,None]-OM[None,:])**2/OV[None,:]
    lp-=logsumexp(lp,axis=1,keepdims=True)
    k=(rng.random(len(z))[:,None]>np.cumsum(np.exp(lp),axis=1)).sum(1); k=np.minimum(k,9)
    ys=z-OM[k]; vv=OV[k].copy(); ys[:skip]=np.nan; vv[:skip]=np.nan
    return np.exp(np.clip(rwffbs(ys,vv,gamma,rng,m0=0.,C0=C0),-20,12)/2)

def update_sd(dx,nu,s2,rng):
    a=np.asarray(dx,float); was1=a.ndim==1
    if was1:a=a[:,None]
    count=np.sum(np.isfinite(a),0); ss=np.nansum(a*a,0)
    out=np.atleast_1d(np.sqrt(invgamma.rvs((nu+count)/2,scale=(nu*s2+ss)/2,random_state=rng)))
    return float(out[0]) if was1 else out

def scales(z,ps,rng,skip):
    prior=np.r_[ps,np.repeat((1-ps)/39,39)]
    lp=np.log(np.maximum(prior,1e-15))[None,:]-np.log(SGRID)[None,:]-.5*(z[:,None]/SGRID[None,:])**2
    lp-=logsumexp(lp,axis=1,keepdims=True)
    k=(rng.random(len(z))[:,None]>np.cumsum(np.exp(lp),axis=1)).sum(1)
    s=SGRID[np.minimum(k,39)]; s[:skip]=1.; return s

def draw_ps(s,a,b,rng):
    ordinary=np.isclose(s,1).sum(); return float(rng.beta(a+ordinary,b+s.size-ordinary))

def invertible(theta):
    if np.allclose(theta,0): return theta
    roots=np.roots(np.r_[theta[::-1],1.]); outside=np.abs(roots)>1; roots[outside]=1/roots[outside]
    adjusted=np.real(np.poly(roots)[1:]); return np.pad(adjusted,(0,max(0,len(theta)-len(adjusted))))[:len(theta)]

def draw_theta(target,X,prec,rng):
    P=np.diag(prec*np.arange(1,X.shape[1]+1)**2)+X.T@X; C=np.linalg.pinv(P)
    return invertible(mvdraw(C@X.T@target,C,rng))

def indices(n,q=3):
    ti=np.arange(1,n+1); ec=n+1; cur=n+2; ei=[]
    for i in range(n): ei.append(np.arange(cur,cur+q+1)); cur+=q+1
    return {"tc":0,"ti":ti,"ec":ec,"ei":ei,"k":cur,"nshock":2*(n+1)}

def transition(ix,n):
    F=np.zeros((ix["k"],ix["k"])); F[ix["tc"],ix["tc"]]=1; F[ix["ti"],ix["ti"]]=1
    for ids in ix["ei"]:F[np.ix_(ids[1:],ids[:-1])]=np.eye(len(ids)-1)
    G=np.zeros((ix["k"],ix["nshock"])); G[ix["tc"],0]=1
    for i in range(n):G[ix["ti"][i],1+i]=1
    G[ix["ec"],n+1]=1
    for i,ids in enumerate(ix["ei"]):G[ids[0],n+2+i]=1
    return F,G

def measurement(at,ae,theta,ix):
    T,n=at.shape; out=[]
    for t in range(T):
        Z=np.zeros((n,ix["k"])); Z[:,ix["tc"]]=at[t]; Z[np.arange(n),ix["ti"]]=1; Z[:,ix["ec"]]=ae[t]
        for i,ids in enumerate(ix["ei"]):Z[i,ids]=np.r_[1,theta[i]]
        out.append(Z)
    return out

def ffbs(Y,Z,F,G,sd,rng):
    T,n=Y.shape;k=F.shape[0]; mf=np.zeros((T,k));mp=np.zeros((T,k));Cf=np.zeros((T,k,k));Cp=np.zeros((T,k,k))
    m=np.zeros(k);C=np.zeros((k,k)); ids=np.setdiff1d(np.arange(k),[0,n+1]); C[ids,ids]=1; C[np.arange(1,n+1),np.arange(1,n+1)]=11
    R=1e-6*np.eye(n)
    for t in range(T):
        a=F@m; P=sym(F@C@F.T+(G*(sd[t]**2)[None,:])@G.T); v=Y[t]-Z[t]@a; S=sym(Z[t]@P@Z[t].T+R)
        K=solve_sym(S,Z[t]@P).T; m=a+K@v; C=sym(P-K@Z[t]@P)
        mp[t],Cp[t],mf[t],Cf[t]=a,P,m,C
    x=np.zeros((T,k));x[-1]=mvdraw(mf[-1],Cf[-1],rng)
    for t in range(T-2,-1,-1):
        J=solve_sym(Cp[t+1],F@Cf[t]).T; mu=mf[t]+J@(x[t+1]-mp[t+1]); V=sym(Cf[t]-J@Cp[t+1]@J.T)
        x[t]=mvdraw(mu,V,rng)
    return x

def shared_loading_stats(response,H,theta,loading_sd,idio_sd):
    # Gaussian likelihood for the shared initial transitory-loading level.
    T=len(response);q=len(theta);k=q+3;shock=np.arange(2,k)
    F=np.zeros((k,k));F[0,0]=1.;F[1,1]=1.;F[np.ix_(shock[1:],shock[:-1])]=np.eye(q)
    m0=np.zeros(k);m1=np.zeros(k);C=np.diag(np.r_[11.,1.,np.ones(q+1)])
    precision=0.;score=0.
    for t in range(T):
        if t==0:
            a0=m0;a1=m1;P=C
        else:
            Q=np.zeros((k,k));Q[0,0]=max(loading_sd[0],1e-8)**2;Q[1,1]=max(loading_sd[1],1e-8)**2;Q[2,2]=max(idio_sd[t],1e-8)**2
            a0=F@m0;a1=F@m1;P=sym(F@C@F.T+Q)
        h=np.r_[H[t],1.,theta];S=float(h@P@h+1e-6);v0=response[t]-h@a0;d=H[t,1]+h@a1
        precision+=d*d/S;score+=d*v0/S;K=P@h/S
        m0=a0+K*v0;m1=a1-K*d;C=sym(P-np.outer(K,h)@P)
    return precision,score

def loading_ffbs(response,H,theta,loading_sd,idio_sd,rng):
    # Draw sector loading deviations and MA shock states conditional on the shared level.
    T=len(response);q=len(theta);k=q+3;shock=np.arange(2,k)
    F=np.zeros((k,k));F[0,0]=1.;F[1,1]=1.;F[np.ix_(shock[1:],shock[:-1])]=np.eye(q)
    m=np.zeros(k);C=np.diag(np.r_[11.,1.,np.ones(q+1)])
    mf=np.zeros((T,k));mp=np.zeros((T,k));Cf=np.zeros((T,k,k));Cp=np.zeros((T,k,k))
    for t in range(T):
        if t==0:
            a=m;P=C
        else:
            Q=np.zeros((k,k));Q[0,0]=max(loading_sd[0],1e-8)**2;Q[1,1]=max(loading_sd[1],1e-8)**2;Q[2,2]=max(idio_sd[t],1e-8)**2
            a=F@m;P=sym(F@C@F.T+Q)
        h=np.r_[H[t],1.,theta];S=float(h@P@h+1e-6);K=P@h/max(S,1e-12)
        m=a+K*(response[t]-h@a);C=sym(P-np.outer(K,h)@P);mp[t],Cp[t],mf[t],Cf[t]=a,P,m,C
    x=np.zeros((T,k));x[-1]=mvdraw(mf[-1],Cf[-1],rng)
    for t in range(T-2,-1,-1):
        J=solve_sym(Cp[t+1],F@Cf[t]).T;mu=mf[t]+J@(x[t+1]-mp[t+1]);V=sym(Cf[t]-J@Cp[t+1]@J.T)
        x[t]=mvdraw(mu,V,rng)
    return x[:,:2]

def ma_effect(e,theta):
    out=e.copy()
    for i in range(e.shape[1]):
        for lag in range(1,theta.shape[1]+1):out[lag:,i]+=theta[i,lag-1]*e[:-lag,i]
    return out

## Estimate volatility

Progress is reported every 25 iterations with an updated time-remaining estimate. As in the NY Fed sampler, each loading draw integrates over the sector's idiosyncratic MA shocks before the full state vector is redrawn. The sampler retains the common and sector-specific variance aggregates for the MCT volatility measure, plus the 17 sector contributions at the latest date.

In [ ]:
def estimate_model(Y,W,burn,draws,seed):
    rng=np.random.default_rng(seed)
    T,n=Y.shape
    q=3
    prec_ma=.1
    nu_lam=12.
    s2_lam=.25**2/60/12
    nu_gam=60.
    s2_gam=1/60/12
    a_ps=(1-1/48)*120
    b_ps=(1/48)*120
    ix=indices(n,q)
    F,G=transition(ix,n)
    ys=np.std(Y,0)
    at=np.tile(ys/16,(T,1));ae=at.copy();sdtc=np.ones(T);sdti=np.tile(ys/16,(T,1))
    sdec=np.ones(T);sdei=np.tile(ys/16,(T,1));sc=np.ones(T);si=np.ones((T,n));theta=np.zeros((n,q))
    lt=np.repeat(np.sqrt(s2_lam),n);le=lt.copy();gdtc=np.sqrt(s2_gam);gdti=np.repeat(gdtc,n)
    gec=gdtc;gei=np.repeat(gdtc,n);psc=a_ps/(a_ps+b_ps);psi=np.repeat(psc,n);skip=4
    keep={"common_var":[],"idio_var":[],"latest_common_sector_var":[],"latest_idio_sector_var":[]}
    sd=np.c_[sdtc,sdti,sdec*sc,sdei*si]
    st=ffbs(Y,measurement(at,ae,theta,ix),F,G,sd,rng)
    tc=st[:,ix["tc"]];ti=st[:,ix["ti"]];ec=st[:,ix["ec"]]
    total=burn+draws;started=time.time()
    for it in range(total):
        H=np.c_[tc,ec]
        shared_precision=.1;shared_score=0.
        for i in range(n):
            p,s=shared_loading_stats(Y[:,i]-ti[:,i],H,theta[i],np.array([lt[i],le[i]]),sdei[:,i]*si[:,i])
            shared_precision+=p;shared_score+=s
        shared_var=1/shared_precision;shared_level=rng.normal(shared_var*shared_score,np.sqrt(shared_var))
        for i in range(n):
            response=Y[:,i]-ti[:,i]-H[:,1]*shared_level
            a=loading_ffbs(response,H,theta[i],np.array([lt[i],le[i]]),sdei[:,i]*si[:,i],rng)
            at[:,i]=a[:,0];ae[:,i]=shared_level+a[:,1]
        lt=update_sd(np.diff(at[skip:],axis=0),nu_lam,s2_lam,rng)
        le=update_sd(np.diff(ae[skip:],axis=0),nu_lam,s2_lam,rng)
        sd=np.c_[sdtc,sdti,sdec*sc,sdei*si]
        st=ffbs(Y,measurement(at,ae,theta,ix),F,G,sd,rng)
        tc=st[:,ix["tc"]];ti=st[:,ix["ti"]];ec=st[:,ix["ec"]]
        ei=np.column_stack([st[:,z[0]] for z in ix["ei"]]);mai=ma_effect(ei,theta)
        for i in range(n):
            rows=np.arange(skip,T)
            X=np.column_stack([ei[rows-l,i] for l in range(1,q+1)])
            scl=np.maximum(sdei[rows,i]*si[rows,i],1e-8)
            theta[i]=draw_theta(mai[rows,i]/scl,X/scl[:,None],prec_ma,rng)
        dtc=np.r_[np.nan,np.diff(tc)]
        dti=np.vstack([np.full((1,n),np.nan),np.diff(ti,axis=0)])
        sdtc=update_sv(dtc,sdtc,gdtc,rng,skip,C0=0.)
        for i in range(n):sdti[:,i]=update_sv(dti[:,i],sdti[:,i],gdti[i],rng,skip,C0=100.)
        sdec=update_sv(ec/np.maximum(sc,1),sdec,gec,rng,skip,C0=0.)
        for i in range(n):sdei[:,i]=update_sv(ei[:,i]/np.maximum(si[:,i],1),sdei[:,i],gei[i],rng,skip,C0=100.)
        gdtc=update_sd(2*np.diff(np.log(sdtc[skip:])),nu_gam,s2_gam,rng)
        gdti=update_sd(2*np.diff(np.log(sdti[skip:]),axis=0),nu_gam,s2_gam,rng)
        gec=update_sd(2*np.diff(np.log(sdec[skip:])),nu_gam,s2_gam,rng)
        gei=update_sd(2*np.diff(np.log(sdei[skip:]),axis=0),nu_gam,s2_gam,rng)
        sc=scales(ec/np.maximum(sdec,1e-8),psc,rng,skip)
        psc=draw_ps(sc[skip:],a_ps,b_ps,rng)
        for i in range(n):
            si[:,i]=scales(ei[:,i]/np.maximum(sdei[:,i],1e-8),psi[i],rng,skip)
            psi[i]=draw_ps(si[skip:,i],a_ps,b_ps,rng)
        if it>=burn:
            ma_factor=1+np.sum(theta**2,axis=1)
            weighted_loading=W*ae
            aggregate_loading=weighted_loading.sum(axis=1)
            common_var=(aggregate_loading*sdec)**2
            idio_sector_var=ma_factor[None,:]*(W*sdei)**2
            idio_var=idio_sector_var.sum(axis=1)
            latest_common_sector_var=weighted_loading[-1]*aggregate_loading[-1]*sdec[-1]**2
            keep["common_var"].append(common_var.astype(np.float32))
            keep["idio_var"].append(idio_var.astype(np.float32))
            keep["latest_common_sector_var"].append(latest_common_sector_var.astype(np.float32))
            keep["latest_idio_sector_var"].append(idio_sector_var[-1].astype(np.float32))
        if it==0 or (it+1)%25==0 or (it+1)==total:
            elapsed=time.time()-started
            remaining=elapsed/(it+1)*(total-it-1)
            print(f"Draw {it+1}/{total} | elapsed {elapsed/3600:.1f}h | remaining {remaining/3600:.1f}h")
    return {name:np.stack(values) for name,values in keep.items()}


POST=estimate_model(y,w,BURN,DRAWS,SEED)

## Summarize and save the volatility measure

In [ ]:
common_variance=POST["common_var"].astype(float)
sector_variance=POST["idio_var"].astype(float)
total_variance=common_variance+sector_variance

common_volatility=np.sqrt(np.maximum(common_variance,0))
sector_volatility=np.sqrt(np.maximum(sector_variance,0))
total_volatility=np.sqrt(np.maximum(total_variance,0))
common_share=common_variance/np.maximum(total_variance,1e-16)

probabilities=[1/6,0.5,5/6]
result=pd.DataFrame({"date":dates})
for name,posterior_draws in {
    "total_volatility":total_volatility,
    "common_volatility":common_volatility,
    "sector_specific_volatility":sector_volatility,
    "common_variance_share":common_share,
}.items():
    lower,median,upper=np.quantile(posterior_draws,probabilities,axis=0)
    result[f"{name}_lower"]=lower
    result[f"{name}_median"]=median
    result[f"{name}_upper"]=upper

result["units"]="annualized percentage points"
result["data_vintage"]=data["vintage"]
result["retrieved_utc"]=data["retrieved_utc"]
result.to_csv(OUTPUT/"estimates"/"mct_core_pce_volatility_fast.csv",index=False)

np.savez_compressed(
    OUTPUT/"estimates"/"mct_volatility_draws.npz",
    dates=dates.astype(str).to_numpy(),
    common_variance=POST["common_var"],
    sector_specific_variance=POST["idio_var"],
    latest_common_sector_variance=POST["latest_common_sector_var"],
    latest_sector_specific_variance=POST["latest_idio_sector_var"],
    seed=np.array(SEED),
    data_vintage=np.array(data["vintage"]),
)

if not np.isfinite(result.filter(like="volatility").dropna().to_numpy()).all():
    raise RuntimeError("Non-finite volatility estimates were produced.")
result.tail(3)

## Chart

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(12,8),sharex=True)

axes[0].fill_between(result.date,result.total_volatility_lower,result.total_volatility_upper,color="tab:blue",alpha=.2,label="66.7% interval")
axes[0].plot(result.date,result.total_volatility_median,color="tab:blue",lw=1.8,label="Total")
axes[0].plot(result.date,result.common_volatility_median,color="tab:orange",lw=1.2,label="Common")
axes[0].plot(result.date,result.sector_specific_volatility_median,color="tab:green",lw=1.2,label="Sector-specific")
axes[0].set_title("MCT core PCE inflation volatility")
axes[0].set_ylabel("Annualized percentage points")
axes[0].legend(ncol=4,fontsize=8)

axes[1].plot(result.date,result.common_variance_share_median,color="tab:red",lw=1.6,label="Common share")
axes[1].plot(result.date,1-result.common_variance_share_median,color="tab:purple",lw=1.3,label="Sector-specific share")
axes[1].set_ylim(0,1)
axes[1].set_title("Variance decomposition")
axes[1].set_ylabel("Share")
axes[1].legend()

fig.tight_layout()
fig.savefig(OUTPUT/"figures"/"mct_core_pce_volatility_fast.png",dpi=160,bbox_inches="tight")
plt.show()

## Latest sector contribution to total variance

For the common component, each sector receives its Euler contribution to aggregate common variance. The sector-specific component is its weighted idiosyncratic variance, including its MA(3) adjustment. These 17 contributions add exactly to total MCT variance within every posterior draw.

A common contribution can be negative when a sector's common-shock loading offsets the aggregate loading. The error bars show the central 66.7% posterior interval.

In [ ]:
latest_common_sector=POST["latest_common_sector_var"].astype(float)
latest_idio_sector=POST["latest_idio_sector_var"].astype(float)
latest_total_sector=latest_common_sector+latest_idio_sector
latest_total_variance=POST["common_var"][:,-1].astype(float)+POST["idio_var"][:,-1].astype(float)

identity_error=np.max(np.abs(latest_total_sector.sum(axis=1)-latest_total_variance))
if identity_error>1e-5:
    raise RuntimeError("Sector contributions do not add to total variance.")

probabilities=[1/6,0.5,5/6]
common_lower,common_median,common_upper=np.quantile(latest_common_sector,probabilities,axis=0)
idio_lower,idio_median,idio_upper=np.quantile(latest_idio_sector,probabilities,axis=0)
total_lower,total_median,total_upper=np.quantile(latest_total_sector,probabilities,axis=0)
share_draws=latest_total_sector/np.maximum(latest_total_variance[:,None],1e-16)

latest_sector=pd.DataFrame({
    "date":dates[-1],
    "sector_id":SECTOR_IDS,
    "sector_label":SECTOR_LABELS,
    "included_in_core":CORE,
    "common_variance_contribution_lower":common_lower,
    "common_variance_contribution_median":common_median,
    "common_variance_contribution_upper":common_upper,
    "sector_specific_variance_contribution_lower":idio_lower,
    "sector_specific_variance_contribution_median":idio_median,
    "sector_specific_variance_contribution_upper":idio_upper,
    "total_variance_contribution_lower":total_lower,
    "total_variance_contribution_median":total_median,
    "total_variance_contribution_upper":total_upper,
    "posterior_mean_share_of_total_variance":share_draws.mean(axis=0),
    "posterior_probability_positive":(latest_total_sector>0).mean(axis=0),
})
latest_sector["units"]="squared annualized percentage points"
latest_sector["data_vintage"]=data["vintage"]
latest_sector=latest_sector.sort_values("total_variance_contribution_median",ascending=False).reset_index(drop=True)
latest_sector.to_csv(OUTPUT/"estimates"/"latest_sector_variance_contributions.csv",index=False)

plot_data=latest_sector.sort_values("total_variance_contribution_median")
median=plot_data["total_variance_contribution_median"].to_numpy()
lower=plot_data["total_variance_contribution_lower"].to_numpy()
upper=plot_data["total_variance_contribution_upper"].to_numpy()
colors=np.where(median>=0,"tab:blue","tab:red")

fig,ax=plt.subplots(figsize=(11,8))
ax.barh(
    plot_data["sector_label"],median,
    xerr=np.vstack([median-lower,upper-median]),
    color=colors,alpha=.82,ecolor="0.35",capsize=2,
)
ax.axvline(0,color="black",lw=.8)
ax.set_title(f"Latest sector contribution to total core PCE variance ({dates[-1]:%B %Y})")
ax.set_xlabel("Squared annualized percentage points")
fig.tight_layout()
fig.savefig(OUTPUT/"figures"/"latest_sector_variance_contributions.png",dpi=160,bbox_inches="tight")
plt.show()

latest_sector[[
    "sector_label","total_variance_contribution_median",
    "total_variance_contribution_lower","total_variance_contribution_upper",
    "posterior_mean_share_of_total_variance","posterior_probability_positive",
]]

## Interpretation

The headline series is `total_volatility_median`. `common_volatility_median` measures volatility associated with shocks shared across sectors; `sector_specific_volatility_median` measures the volatility of shocks that diversify across sectors.

Each run pulls the latest revised BEA history, so the complete estimated path can change after a BEA revision—not only the newest observation. The model and priors follow the MCT approach, while 500 retained draws provide less Monte Carlo precision than a 3,000-draw publication run.

The latest-sector table decomposes variance, not the inflation rate. A large positive contribution identifies a sector that materially raises uncertainty in aggregate core PCE inflation at the latest date; a negative common contribution means the sector partly offsets the aggregate common shock.